# Feature attribution and stability

Cross-check LightGBM gain and split counts with held-out permutation importance and mean absolute SHAP. Agreement is useful evidence; disagreement is a reason to inspect leakage, redundancy, and instability rather than blindly deleting a feature.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
latest = json.loads((ROOT / "reports/lol/training/latest.json").read_text())
run = Path(latest['report_directory'])
if not run.is_absolute():
    run = ROOT / run
models = [path for path in run.iterdir() if path.is_dir() and (path / 'feature_importances.parquet').exists()]
[path.name for path in models]

In [ ]:
def attribution_table(model_dir: Path) -> pd.DataFrame:
    base = pd.read_parquet(model_dir / 'feature_importances.parquet')
    for filename in ('permutation_importances.parquet', 'shap_importances.parquet'):
        path = model_dir / filename
        if path.exists():
            base = base.merge(pd.read_parquet(path), on='feature', how='left', validate='one_to_one')
    return base

tables = {path.name: attribution_table(path) for path in models}
display(tables[next(iter(tables))].head(30))

In [ ]:
for name, table in tables.items():
    columns = [col for col in ('gain_fraction', 'importance_mean', 'mean_absolute_shap') if col in table]
    ranked = table.set_index('feature')[columns].rank(ascending=False)
    display(name, ranked.corr(method='spearman'))
    table.nlargest(20, 'gain').sort_values('gain').plot.barh(x='feature', y='gain', title=f'{name}: gain')